# Real-time pipeline threshold tuning

This notebook finds practical thresholds for the live drone pipeline:

- Human detector: choose the threshold with the highest recall while precision stays at least `0.90`.
- Posture classifier: use fixed `0.50` unknown confidence, then report what that means for coverage and accepted-label accuracy.

Use validation/test data that looks as close as possible to the real drone camera feed.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from ultralytics import YOLO

try:
    import yaml
except ImportError as exc:
    raise ImportError("Install PyYAML first: pip install PyYAML") from exc

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DETECTOR_WEIGHTS = PROJECT_ROOT / "runs" / "human_detection" / "weights" / "best.pt"
DETECTION_DATA_YAML = PROJECT_ROOT / "data" / "labeled" / "human_detection" / "data.yaml"

CLASSIFIER_WEIGHTS = PROJECT_ROOT / "runs" / "posture_classification" / "mobilenet_v3_small" / "best.pt"
CLASS_MAP_PATH = PROJECT_ROOT / "runs" / "posture_classification" / "mobilenet_v3_small" / "class_to_idx.json"
CLASSIFICATION_DATA_DIR = PROJECT_ROOT / "data" / "labeled" / "posture_classification"

OUTPUT_DIR = PROJECT_ROOT / "runs" / "threshold_tuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_DETECTOR_PRECISION = 0.90
MIN_POSTURE_ACCEPTED_ACCURACY = 0.90
IOU_MATCH_THRESHOLD = 0.50
DETECTOR_IMAGE_SIZE = 320
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Detector weights:", DETECTOR_WEIGHTS, DETECTOR_WEIGHTS.exists())
print("Classifier weights:", CLASSIFIER_WEIGHTS, CLASSIFIER_WEIGHTS.exists())
print("Output dir:", OUTPUT_DIR)

## Human detector threshold

The detector threshold is selected by sweeping confidence values. For each confidence value, predictions are matched to ground-truth boxes with IoU `>= 0.50`, then precision and recall are calculated.

Selection rule:

`best = highest recall where precision >= 0.90`

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _existing_path_candidates(path_value, yaml_path):
    raw = Path(path_value)
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([
            (yaml_path.parent / raw).resolve(),
            (yaml_path.parent.parent / raw).resolve(),
            (PROJECT_ROOT / raw).resolve(),
        ])
    return [p for p in candidates if p.exists()]


def find_detection_split(yaml_path: Path, preferred=("test", "val", "valid", "train")):
    with yaml_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    split_aliases = {
        "test": ["test"],
        "val": ["val", "valid"],
        "valid": ["valid", "val"],
        "train": ["train"],
    }

    checked = []
    for desired in preferred:
        for key in split_aliases.get(desired, [desired]):
            if key not in data:
                continue
            for image_dir in _existing_path_candidates(data[key], yaml_path):
                checked.append(str(image_dir))
                label_dir = Path(str(image_dir).replace(f"{Path('images')}", f"{Path('labels')}"))
                if image_dir.exists() and label_dir.exists():
                    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
                    if images:
                        return key, image_dir, label_dir

    base = yaml_path.parent
    manual_names = ["test", "valid", "val", "train"]
    for name in manual_names:
        for root in [base, base.parent, PROJECT_ROOT / "data" / "processed"]:
            candidates = [root / name / "images", root / "images" / name]
            for image_dir in candidates:
                label_dir = Path(str(image_dir).replace(f"{Path('images')}", f"{Path('labels')}"))
                checked.append(str(image_dir))
                if image_dir.exists() and label_dir.exists():
                    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
                    if images:
                        return name, image_dir, label_dir

    raise FileNotFoundError("Could not find a detection image/label split. Checked:\n" + "\n".join(checked))


def yolo_box_to_xyxy(values, width, height):
    xc, yc, bw, bh = values
    x1 = (xc - bw / 2) * width
    y1 = (yc - bh / 2) * height
    x2 = (xc + bw / 2) * width
    y2 = (yc + bh / 2) * height
    return [x1, y1, x2, y2]


def yolo_segment_to_xyxy(values, width, height):
    points = np.array(values, dtype=np.float32).reshape(-1, 2)
    x1 = float(points[:, 0].min() * width)
    y1 = float(points[:, 1].min() * height)
    x2 = float(points[:, 0].max() * width)
    y2 = float(points[:, 1].max() * height)
    return [x1, y1, x2, y2]


def load_yolo_labels(label_path: Path, width: int, height: int):
    if not label_path.exists():
        return np.zeros((0, 4), dtype=np.float32)
    boxes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        values = [float(x) for x in parts[1:]]
        if len(values) == 4:
            boxes.append(yolo_box_to_xyxy(values, width, height))
        elif len(values) >= 6 and len(values) % 2 == 0:
            boxes.append(yolo_segment_to_xyxy(values, width, height))
        else:
            warnings.warn(f"Unsupported label row in {label_path.name}: {line[:80]}")
    return np.array(boxes, dtype=np.float32)


def box_iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    inter_x1 = np.maximum(ax1, bx1)
    inter_y1 = np.maximum(ay1, by1)
    inter_x2 = np.minimum(ax2, bx2)
    inter_y2 = np.minimum(ay2, by2)
    inter = np.maximum(inter_x2 - inter_x1, 0) * np.maximum(inter_y2 - inter_y1, 0)
    area_a = np.maximum(ax2 - ax1, 0) * np.maximum(ay2 - ay1, 0)
    area_b = np.maximum(bx2 - bx1, 0) * np.maximum(by2 - by1, 0)
    union = area_a + area_b - inter
    return inter / np.maximum(union, 1e-9)


split_name, det_image_dir, det_label_dir = find_detection_split(DETECTION_DATA_YAML)
det_images = sorted(p for p in det_image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
print(f"Using detection split: {split_name}")
print("Images:", len(det_images))
print("Image dir:", det_image_dir)
print("Label dir:", det_label_dir)

In [ ]:
detector = YOLO(str(DETECTOR_WEIGHTS))
detector_records = []

for image_path in det_images:
    image = cv2.imread(str(image_path))
    if image is None:
        warnings.warn(f"Could not read image: {image_path}")
        continue
    height, width = image.shape[:2]
    label_path = det_label_dir / f"{image_path.stem}.txt"
    gt_boxes = load_yolo_labels(label_path, width, height)

    result = detector.predict(
        source=image,
        imgsz=DETECTOR_IMAGE_SIZE,
        conf=0.001,
        iou=0.70,
        device=DEVICE,
        verbose=False,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        pred_boxes = np.zeros((0, 4), dtype=np.float32)
        pred_conf = np.zeros((0,), dtype=np.float32)
    else:
        pred_boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
        pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)

    detector_records.append({
        "image": str(image_path),
        "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes,
        "pred_conf": pred_conf,
    })

print("Images evaluated:", len(detector_records))

In [ ]:
def evaluate_detector_threshold(records, threshold, iou_threshold=0.50):
    tp = fp = fn = 0
    for record in records:
        gt_boxes = record["gt_boxes"]
        keep = record["pred_conf"] >= threshold
        pred_boxes = record["pred_boxes"][keep]
        pred_conf = record["pred_conf"][keep]

        if len(pred_boxes):
            order = np.argsort(-pred_conf)
            pred_boxes = pred_boxes[order]

        matched_gt = set()
        ious = box_iou_matrix(pred_boxes, gt_boxes)
        for pred_idx in range(len(pred_boxes)):
            if len(gt_boxes) == 0:
                fp += 1
                continue
            best_gt = int(np.argmax(ious[pred_idx]))
            best_iou = float(ious[pred_idx, best_gt])
            if best_iou >= iou_threshold and best_gt not in matched_gt:
                tp += 1
                matched_gt.add(best_gt)
            else:
                fp += 1
        fn += len(gt_boxes) - len(matched_gt)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    beta = 2.0
    f2 = (1 + beta ** 2) * precision * recall / (beta ** 2 * precision + recall) if precision + recall else 0.0
    return {
        "threshold": threshold,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
    }


thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)
detector_threshold_df = pd.DataFrame(
    [evaluate_detector_threshold(detector_records, float(t), IOU_MATCH_THRESHOLD) for t in thresholds]
)

def choose_detector_threshold(df, min_precision=0.90):
    candidates = df[df["precision"] >= min_precision]
    if len(candidates):
        row = candidates.sort_values(
            ["recall", "f1", "precision", "threshold"], ascending=[False, False, False, True]
        ).iloc[0]
        return row, True
    row = df.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]
    return row, False


best_detector_row, detector_threshold_meets_precision = choose_detector_threshold(
    detector_threshold_df, MIN_DETECTOR_PRECISION
)

detector_tradeoffs = []
for precision_floor in [0.95, 0.90, 0.85, 0.80, 0.75]:
    row, found = choose_detector_threshold(detector_threshold_df, precision_floor)
    detector_tradeoffs.append({
        "precision_floor": precision_floor,
        "found": found,
        "threshold": float(row["threshold"]),
        "precision": float(row["precision"]),
        "recall": float(row["recall"]),
        "f1": float(row["f1"]),
        "f2": float(row["f2"]),
    })
detector_tradeoff_df = pd.DataFrame(detector_tradeoffs)

detector_threshold_df.to_csv(OUTPUT_DIR / "detector_threshold_sweep.csv", index=False)
best_detector_row.to_frame().T.to_csv(OUTPUT_DIR / "best_detector_threshold.csv", index=False)
detector_tradeoff_df.to_csv(OUTPUT_DIR / "detector_precision_floor_tradeoffs.csv", index=False)

print("Meets preferred precision floor:", detector_threshold_meets_precision)
print(best_detector_row)
display(detector_tradeoff_df)
detector_threshold_df.head()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(detector_threshold_df["threshold"], detector_threshold_df["precision"], label="precision")
plt.plot(detector_threshold_df["threshold"], detector_threshold_df["recall"], label="recall")
plt.axhline(MIN_DETECTOR_PRECISION, color="tab:red", linestyle="--", label="minimum precision")
plt.axvline(float(best_detector_row["threshold"]), color="tab:green", linestyle="--", label="selected threshold")
plt.xlabel("Detector confidence threshold")
plt.ylabel("Score")
plt.title("Human detector threshold sweep")
plt.ylim(0, 1.02)
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "detector_threshold_sweep.png", dpi=160)
plt.show()

## Posture classifier fixed unknown threshold

For posture classification, the live rule is intentionally simple: mark the crop as `unknown` only when the top class confidence is below `0.50`.

Selection rule:

`unknown = top1_confidence < 0.50`

In [ ]:
def find_classification_images(root: Path, preferred=("test", "valid", "val", "train")):
    if not root.exists():
        raise FileNotFoundError(f"Classification data folder not found: {root}")

    for split in preferred:
        split_dir = root / split
        if split_dir.exists():
            rows = []
            for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
                for image_path in sorted(class_dir.rglob("*")):
                    if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                        rows.append({"image": image_path, "class_name": class_dir.name, "split": split})
            if rows:
                return pd.DataFrame(rows)

    rows = []
    ignored = {"train", "test", "valid", "val", "images", "labels"}
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir() and p.name not in ignored):
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({"image": image_path, "class_name": class_dir.name, "split": "all"})
    if rows:
        return pd.DataFrame(rows)

    raise FileNotFoundError(f"No classification images found under {root}")


def build_mobilenet(num_classes: int):
    model = models.mobilenet_v3_small(weights=None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


def load_classifier(weights_path: Path, class_map_path: Path, device: str):
    class_to_idx = json.loads(class_map_path.read_text(encoding="utf-8"))
    idx_to_class = {idx: name for name, idx in class_to_idx.items()}
    model = build_mobilenet(len(class_to_idx))
    checkpoint = torch.load(weights_path, map_location=device)
    if isinstance(checkpoint, dict):
        state = checkpoint.get("model_state_dict") or checkpoint.get("state_dict") or checkpoint
    else:
        state = checkpoint
    state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(device).eval()
    return model, class_to_idx, idx_to_class


posture_rows = find_classification_images(CLASSIFICATION_DATA_DIR)
classifier, class_to_idx, idx_to_class = load_classifier(CLASSIFIER_WEIGHTS, CLASS_MAP_PATH, DEVICE)
posture_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Posture images:", len(posture_rows))
print("Split used:", posture_rows["split"].iloc[0])
print("Classes:", class_to_idx)

In [ ]:
posture_predictions = []

with torch.no_grad():
    for row in posture_rows.itertuples(index=False):
        image = Image.open(row.image).convert("RGB")
        x = posture_tf(image).unsqueeze(0).to(DEVICE)
        logits = classifier(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
        order = np.argsort(-probs)
        top1_idx = int(order[0])
        top2_idx = int(order[1]) if len(order) > 1 else top1_idx
        posture_predictions.append({
            "image": str(row.image),
            "true_class": row.class_name,
            "pred_class": idx_to_class[top1_idx],
            "confidence": float(probs[top1_idx]),
            "top2_class": idx_to_class[top2_idx],
            "top2_confidence": float(probs[top2_idx]),
            "margin": float(probs[top1_idx] - probs[top2_idx]),
            "correct": row.class_name == idx_to_class[top1_idx],
        })

posture_pred_df = pd.DataFrame(posture_predictions)
posture_pred_df.to_csv(OUTPUT_DIR / "posture_classifier_predictions.csv", index=False)
posture_pred_df.head()

In [ ]:
def evaluate_posture_threshold(df, threshold):
    accepted = df[df["confidence"] >= threshold]
    coverage = len(accepted) / len(df) if len(df) else 0.0
    accepted_accuracy = accepted["correct"].mean() if len(accepted) else 0.0
    rejected = len(df) - len(accepted)
    return {
        "threshold": threshold,
        "accepted": len(accepted),
        "unknown": rejected,
        "coverage": coverage,
        "accepted_accuracy": accepted_accuracy,
    }


posture_threshold_df = pd.DataFrame(
    [evaluate_posture_threshold(posture_pred_df, float(t)) for t in thresholds]
)
fixed_posture_threshold = 0.50
best_posture_row = posture_threshold_df[
    np.isclose(posture_threshold_df["threshold"], fixed_posture_threshold)
].iloc[0]

posture_threshold_df.to_csv(OUTPUT_DIR / "posture_unknown_threshold_sweep.csv", index=False)
best_posture_row.to_frame().T.to_csv(OUTPUT_DIR / "fixed_posture_unknown_threshold.csv", index=False)

print(best_posture_row)
posture_threshold_df.head()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(posture_threshold_df["threshold"], posture_threshold_df["accepted_accuracy"], label="accepted accuracy")
plt.plot(posture_threshold_df["threshold"], posture_threshold_df["coverage"], label="coverage")
plt.axhline(MIN_POSTURE_ACCEPTED_ACCURACY, color="tab:red", linestyle="--", label="minimum accepted accuracy")
plt.axvline(float(best_posture_row["threshold"]), color="tab:green", linestyle="--", label="selected threshold")
plt.xlabel("Posture confidence threshold")
plt.ylabel("Score")
plt.title("Posture unknown threshold sweep")
plt.ylim(0, 1.02)
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "posture_unknown_threshold_sweep.png", dpi=160)
plt.show()

In [ ]:
summary = {
    "detector_confidence": float(best_detector_row["threshold"]) if detector_threshold_meets_precision else None,
    "selected_detector_confidence": float(best_detector_row["threshold"]),
    "fallback_detector_confidence": 0.35,
    "detector_threshold_meets_precision": detector_threshold_meets_precision,
    "detector_precision": float(best_detector_row["precision"]),
    "detector_recall": float(best_detector_row["recall"]),
    "posture_unknown_confidence": 0.50,
    "posture_accepted_accuracy": float(best_posture_row["accepted_accuracy"]),
    "posture_coverage": float(best_posture_row["coverage"]),
    "posture_margin": 0.0,
}

(OUTPUT_DIR / "recommended_realtime_thresholds.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print(json.dumps(summary, indent=2))
print("\nUse these in the drone GUI:")
if summary["detector_confidence"] is None:
    print("Detector confidence: no valid threshold reached precision target; keep 0.35 and rerun with a better validation split")
else:
    print(f"Detector confidence: {summary['detector_confidence']:.2f}")
print(f"Unknown confidence: {summary['posture_unknown_confidence']:.2f}")
print("Unknown margin: 0.00")